In [16]:
import collections

In [17]:
corpus = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?"
]

In [18]:
print("Training Corpus")
for doc in corpus:
    print(doc)

Training Corpus
This is the first document.
This document is the second document.
And this is the third one.
Is this the first document?


In [19]:
unique_chars=set()
for doc in corpus:
  for char in doc:
    unique_chars.add(char)

In [20]:
unique_chars

{' ',
 '.',
 '?',
 'A',
 'I',
 'T',
 'c',
 'd',
 'e',
 'f',
 'h',
 'i',
 'm',
 'n',
 'o',
 'r',
 's',
 't',
 'u'}

In [21]:
vocab=list(unique_chars)
vocab.sort()
end_of_word="</w>"
vocab.append(end_of_word)

In [22]:
vocab

[' ',
 '.',
 '?',
 'A',
 'I',
 'T',
 'c',
 'd',
 'e',
 'f',
 'h',
 'i',
 'm',
 'n',
 'o',
 'r',
 's',
 't',
 'u',
 '</w>']

In [23]:
## pretokenization
word_splits = {}
for doc in corpus:
  words=doc.split(' ')
  for word in words:
    char_list=list(word) + [end_of_word]
    word_tuple=tuple(char_list)
    if word_tuple not in word_splits:
      word_splits[word_tuple]=0
    word_splits[word_tuple] += 1



In [24]:
word_splits

{('T', 'h', 'i', 's', '</w>'): 2,
 ('i', 's', '</w>'): 3,
 ('t', 'h', 'e', '</w>'): 4,
 ('f', 'i', 'r', 's', 't', '</w>'): 2,
 ('d', 'o', 'c', 'u', 'm', 'e', 'n', 't', '.', '</w>'): 2,
 ('d', 'o', 'c', 'u', 'm', 'e', 'n', 't', '</w>'): 1,
 ('s', 'e', 'c', 'o', 'n', 'd', '</w>'): 1,
 ('A', 'n', 'd', '</w>'): 1,
 ('t', 'h', 'i', 's', '</w>'): 2,
 ('t', 'h', 'i', 'r', 'd', '</w>'): 1,
 ('o', 'n', 'e', '.', '</w>'): 1,
 ('I', 's', '</w>'): 1,
 ('d', 'o', 'c', 'u', 'm', 'e', 'n', 't', '?', '</w>'): 1}

In [25]:
def get_pair_stats(splits):
   """Counts the frequency of adjacent pairs in the word splits."""
    # Initialize a dictionary with default values of 0 to count pairs of symbols.
    # defaultdict: It's like a regular dictionary (dict), but with a key difference.
    # If you try to access or modify a key that doesn't exist, instead of raising a KeyError,
    # it automatically creates that key and assigns it a default value.
    # int: This is the "default factory" you provide when creating the defaultdict. When a new key is created, it needs a default value, defaultdict calls this factory function. int() called with no arguments returns 0.
   pair_counts = collections.defaultdict(int)
   for word_tuple,freq in splits.items():
    symbols = list(word_tuple)
    for i in range(len(symbols)-1):
      pair = (symbols[i],symbols[i+1])
      pair_counts[pair] += freq

   return pair_counts


In [26]:
def merge_pair(pair_to_merge,splits):
# ### Helper Function: `merge_pair`
#
# This function takes a specific pair (`pair_to_merge`) that we want to combine and the current `splits`. It iterates through all the word representations in `splits`, replaces occurrences of the `pair_to_merge` with a new single token (concatenation of the pair), and returns the updated `splits`.
#
# **Input Example:**
# - `pair_to_merge`: `('i', 's')`
# - `splits`: `{('T', 'h', 'i', 's', '</w>'): 2, ('i', 's', '</w>'): 2, ...}`
#
# **Output Example (`new_splits`):**
# - `{('T', 'h', 'is', '</w>'): 2, ('is', '</w>'): 2, ...}` (assuming 'is' is the merged token)

 new_splits = {}
 (first,second) = pair_to_merge
 merged_token = first + second
 for word_tuple,freq in splits.items():
    symbols=list(word_tuple)
    new_symbols=[]
    i=0
    while i < len(symbols):
      if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
        new_symbols.append(merged_token)
        i+=2
      else:
        new_symbols.append(symbols[i])
        i+=1
      new_splits[tuple(new_symbols)] = freq

 return new_splits


In [27]:
# ### Step 3: Iterative BPE Merging Loop
#
# Now we perform the core BPE training. We'll loop for a fixed number of merges (`num_merges`). In each iteration:
# 1. Calculate the frequencies of all adjacent pairs in the current word representations using `get_pair_stats`.
# 2. Find the pair with the highest frequency (`best_pair`).
# 3. Merge this `best_pair` across all word representations using `merge_pair`.
# 4. Add the newly formed token (concatenation of `best_pair`) to our vocabulary (`vocab`).
# 5. Store the merge rule (mapping the pair to the new token) in the `merges` dictionary.
#
# We'll add print statements to observe the state at each step of the loop.

In [28]:
## Initializing BPE-Training Loop
num_merges=15
merges={}

In [29]:
current_splits=word_splits.copy()
print("\n---Starting BPE Merges---")
print(f"Initial Splits:{current_splits}")
print("-" * 30)

for i in range(num_merges):
  print(f"\nMerge Iteration {i+1}/{num_merges}")

   # 1. Calculate Pair Frequencies
  pair_stats = get_pair_stats(current_splits)
  if not pair_stats:
    print("No more pairs to merge.")
    break

   # 2. Find Best Pair
    # The 'max' function iterates over all key-value pairs in the 'pair_stats' dictionary
    # The 'key=pair_stats.get' tells 'max' to use the frequency (value) for comparison, not the pair (key) itself
    # This way, 'max' selects the pair with the highest frequency
  best_pair = max(pair_stats,key=pair_stats.get)
  best_freq=pair_stats[best_pair]
  print(f"Found Best Pair: {best_pair} with Frequency: {best_freq}")

  # 3.Merge the Best Pair
  current_splits = merge_pair(best_pair,current_splits)
  new_token=best_pair[0] + best_pair[1]
  print(f"Merging {best_pair} into '{new_token}'")
  print(f"Splits after merge: {current_splits}")

  # 4.Update vocabulary
  vocab.append(new_token)
  print(f"Updated Vocabulary: {vocab}")

  # 5.Store merge rule
  merges[best_pair]=new_token
  print(f"Updated merges:{merges}")

  print("-" * 30)






---Starting BPE Merges---
Initial Splits:{('T', 'h', 'i', 's', '</w>'): 2, ('i', 's', '</w>'): 3, ('t', 'h', 'e', '</w>'): 4, ('f', 'i', 'r', 's', 't', '</w>'): 2, ('d', 'o', 'c', 'u', 'm', 'e', 'n', 't', '.', '</w>'): 2, ('d', 'o', 'c', 'u', 'm', 'e', 'n', 't', '</w>'): 1, ('s', 'e', 'c', 'o', 'n', 'd', '</w>'): 1, ('A', 'n', 'd', '</w>'): 1, ('t', 'h', 'i', 's', '</w>'): 2, ('t', 'h', 'i', 'r', 'd', '</w>'): 1, ('o', 'n', 'e', '.', '</w>'): 1, ('I', 's', '</w>'): 1, ('d', 'o', 'c', 'u', 'm', 'e', 'n', 't', '?', '</w>'): 1}
------------------------------

Merge Iteration 1/15
Found Best Pair: ('s', '</w>') with Frequency: 8
Merging ('s', '</w>') into 's</w>'
Splits after merge: {('T',): 2, ('T', 'h'): 2, ('T', 'h', 'i'): 2, ('T', 'h', 'i', 's</w>'): 2, ('i',): 3, ('i', 's</w>'): 3, ('t',): 1, ('t', 'h'): 1, ('t', 'h', 'e'): 4, ('t', 'h', 'e', '</w>'): 4, ('f',): 2, ('f', 'i'): 2, ('f', 'i', 'r'): 2, ('f', 'i', 'r', 's'): 2, ('f', 'i', 'r', 's', 't'): 2, ('f', 'i', 'r', 's', 't', '</w

In [ ]:
### Step 4: Review Final Results
#
# After the loop finishes, we can examine the final state:
# - The learned merge rules (`merges`).
# - The final representation of words after merges (`current_splits`).
# - The complete vocabulary (`vocab`) containing initial characters and learned subword tokens.


In [30]:
# --- BPE Merges Complete ---
print("\n--- BPE Merges Complete ---")
print(f"Final Vocabulary Size: {len(vocab)}")
print("\nLearned Merges (Pair -> New Token):")
# Pretty print merges
for pair, token in merges.items():
    print(f"{pair} -> '{token}'")

print("\nFinal Word Splits after all merges:")
print(current_splits)

print("\nFinal Vocabulary (sorted):")
# Sort for consistent viewing
final_vocab_sorted = sorted(list(set(vocab))) # Use set to remove potential duplicates if any step introduced them
print(final_vocab_sorted)


--- BPE Merges Complete ---
Final Vocabulary Size: 35

Learned Merges (Pair -> New Token):
('s', '</w>') -> 's</w>'
('t', 'h') -> 'th'
('d', 'o') -> 'do'
('do', 'c') -> 'doc'
('doc', 'u') -> 'docu'
('i', 'r') -> 'ir'
('docu', 'm') -> 'docum'
('docum', 'e') -> 'docume'
('docume', 'n') -> 'documen'
('th', 'e') -> 'the'
('f', 'ir') -> 'fir'
('documen', 't') -> 'document'
('i', 's</w>') -> 'is</w>'
('o', 'n') -> 'on'
('T', 'h') -> 'Th'

Final Word Splits after all merges:
{('T',): 2, ('Th',): 2, ('Th', 'i'): 2, ('Th', 'is</w>'): 2, ('i',): 3, ('is</w>',): 3, ('t',): 1, ('th',): 1, ('the',): 4, ('the', '</w>'): 4, ('f',): 2, ('f', 'i'): 2, ('fir',): 2, ('fir', 's'): 2, ('fir', 's', 't'): 2, ('fir', 's', 't', '</w>'): 2, ('d',): 1, ('do',): 1, ('doc',): 1, ('docu',): 1, ('docum',): 1, ('docume',): 1, ('documen',): 1, ('document',): 1, ('document', '.'): 2, ('document', '.', '</w>'): 2, ('document', '</w>'): 1, ('s',): 1, ('s', 'e'): 1, ('s', 'e', 'c'): 1, ('s', 'e', 'c', 'o'): 1, ('s', 'e',